# 04 — Eval adapter vs base + submission

Go/no-go: chỉ dùng adapter nếu thắng base trên holdout PA **và** portal PA.

In [ ]:
# === Setup (chạy mỗi lần mở runtime) ===
from google.colab import drive
drive.mount('/content/drive')
DRIVE_ROOT = '/content/drive/MyDrive/DataMate_ZAC2026'   # thư mục Drive chung (Add shortcut to My Drive nếu được share)
REPO_BRANCH = 'feat/slm-agent'
import os, subprocess
if not os.path.exists('/content/AI-Challenge'):
    subprocess.run(['git', 'clone', '--branch', REPO_BRANCH, 'https://github.com/VTin206/AI-Challenge.git', '/content/AI-Challenge'], check=True)
%cd /content/AI-Challenge
!git pull -q
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
SPLITS = ['train', 'public_test']
# === Giải nén dữ liệu từ Drive vào /content/data (đĩa local, nhanh hơn Drive) ===
import zipfile, os
os.makedirs('/content/data', exist_ok=True)
for split in SPLITS:
    if not os.path.exists(f'/content/data/{split}'):
        zipfile.ZipFile(f'{DRIVE_ROOT}/data/{split}/{split}.zip').extractall('/content/data')
!ls /content/data

In [ ]:
!pip install -q -e . vllm

In [ ]:
ADAPTER = f'{DRIVE_ROOT}/models/sft_run1/adapter'
import json, os
HOLDOUT = f'{DRIVE_ROOT}/training/split.json' if os.path.exists(f'{DRIVE_ROOT}/training/split.json') else 'configs/holdout.json'
hold = json.load(open(HOLDOUT))
hold = {'businesses': hold.get('holdout', hold.get('businesses'))}
json.dump(hold, open('/content/holdout.json', 'w'))
print(hold)

In [ ]:
RUN = 'holdout_adapter_k5'
!python scripts/run_split.py --split /content/data/train --out {DRIVE_ROOT}/runs/{RUN} --config configs/slm.json \
    --holdout /content/holdout.json --lora {ADAPTER} --tau 0 --cache-dir /content/cache --resume
!python scripts/tune_tau.py --log {DRIVE_ROOT}/runs/{RUN}/log.jsonl --gold /content/data/train/answers.jsonl

In [ ]:
RUN = 'holdout_base_k5'
!python scripts/run_split.py --split /content/data/train --out {DRIVE_ROOT}/runs/{RUN} --config configs/slm.json \
    --holdout /content/holdout.json --tau 0 --cache-dir /content/cache --resume
!python scripts/tune_tau.py --log {DRIVE_ROOT}/runs/{RUN}/log.jsonl --gold /content/data/train/answers.jsonl

In [ ]:
TAU = 0.6
RUN = 'public_adapter_run1'   # ĐỔI TÊN cho mỗi lần nộp mới; giữ nguyên khi --resume
!python scripts/run_split.py --split /content/data/public_test --out {DRIVE_ROOT}/runs/{RUN} --config configs/slm.json \
    --lora {ADAPTER} --tau {TAU} --cache-dir /content/cache --resume
print('Upload:', f'{DRIVE_ROOT}/runs/{RUN}/submission.jsonl')